# Real-Time Bitcoin Transaction Anomaly Detection using Anthropic Claude

## Introduction

This project builds a real-time Bitcoin transaction anomaly detection system using Anthropic Claude. It fetches live transaction data from the Blockchair API, extracts structural features in real time, and prepares transactions for anomaly analysis and LLM-based explanations.

The system is modular: real-time ingestion → feature engineering → anomaly detection → natural language explanation → alerting → dashboard visualization.

## Objective

- Ingest real-time Bitcoin transaction data from the Blockchair API.
- Extract structural and behavioral features from transactions.
- Apply a two-stage anomaly detection system (rule-based and LLM-based).
- Generate interpretable explanations for flagged transactions using Anthropic Claude.
- Visualize the results through dashboards and integrate alerts.
- Prepare the system for scalable deployment using Spark and AWS.

## Project Workflow

1. Data Ingestion from Blockchair API
2. Feature Engineering and Time Bucketing
3. Anomaly Detection:
   - Rule-based Filtering
   - Claude LLM Analysis
4. Temporal Pattern Detection
5. Alerting and Dashboarding
6. Deployment and Auto-scaling
7. Testing and Final Reporting

## 1. Integrated Real-Time Pipeline Overview

The full pipeline is split across the following modules:

- **extract_and_upload.py**: 
  - Pulls Bitcoin transactions from Blockchair API.
  - Uploads small structured batches to AWS S3 under `/raw/` folder.

- **preprocess_with_spark.py**:
  - Loads raw transactions from S3.
  - Applies feature engineering, windowed aggregation (1min/5min), and synthetic oversampling using PySpark.
  - Saves processed and balanced data to S3 under `/processed/` folder.

- **anomaly_explainer.py**:
  - Pulls processed anomalies from S3.
  - Generates chain-of-thought style human-readable explanations using Anthropic Claude.
  - Uploads final output JSON files to `/explained/` folder in S3.

## 2. High-Level Architecture

Blockchair API → (extract_and_upload.py) → S3 (/raw/)

         ↓
(preprocess_with_spark.py)

         ↓
S3 (/processed/)

         ↓
(anomaly_explainer.py)

         ↓
S3 (/explained/)

## Important Notes Before Running the Pipeline

- Ensure AWS credentials (`aws configure`) are properly set on your machine.
- Ensure the AWS S3 bucket `btc-anomaly` is created and accessible.
- Ensure Anthropic Claude API key is available and exported as environment variable `ANTHROPIC_API_KEY`.
- Install all required Python packages:
    - `boto3`
    - `s3fs`
    - `pyarrow`
    - `anthropic`
    - `pyspark`
- Install Hadoop-AWS support for Spark (`hadoop-aws` JAR) when running Spark jobs.

Running all code cells sequentially will automatically extract data, preprocess it, and generate explanations.

In [6]:
import subprocess

# Step 1: Run data extraction script
try:
    subprocess.run(["python", "../scripts/extract_and_upload.py"], check=True)
    print("Data extraction completed successfully.")
except subprocess.CalledProcessError as e:
    print("Error occurred during data extraction:", e)

Traceback (most recent call last):
  File "/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/notebooks/../scripts/extract_and_upload.py", line 63, in <module>
    run_extraction_loop()
  File "/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/notebooks/../scripts/extract_and_upload.py", line 51, in run_extraction_loop
    time.sleep(INTERVAL_SECONDS)
KeyboardInterrupt


Batch 1
Blockchair error 400: {"data":null,"context":{"code":400,"error":"Limit value can't be larger than 100. Please use export functions to get larger amounts of data.","market_price_usd":94424,"cache":{"live":true,"duration":120,"since":"2025-04-27 03:14:04","until":"2025-04-27 03:16:04","time":null},"api":{"version":"2.0.95-ie","last_major_update":"2022-11-07 02:00:00","next_major_update":"2023-11-12 02:00:00","documentation":"https:\/\/blockchair.com\/api\/docs","notice":"Try out our new API v.3: https:\/\/3xpl.com\/data"},"servers":"API4,BTC5","time":2.5987625122070312e-5,"render_time":0.0027391910552978516,"full_time":0.002765178680419922,"request_cost":5}}
No data fetched.


KeyboardInterrupt: 

In [10]:
# Step 2: Run preprocessing script using spark-submit
try:
    subprocess.run([
        "spark-submit",
        "--packages",
        "org.apache.hadoop:hadoop-aws:3.3.2",
        "../scripts/preprocess_with_spark.py"
    ], check=True)
    print("Data preprocessing completed successfully.")
except subprocess.CalledProcessError as e:
    print("Error occurred during data preprocessing:", e)

25/04/26 23:16:13 WARN Utils: Your hostname, Anveshs-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 10.0.0.124 instead (on interface en0)
25/04/26 23:16:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/chitturi/.ivy2/cache
The jars for the packages stored in: /Users/chitturi/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f3c61268-0cef-4c9b-bb9a-0bbdddbd06d5;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central


:: loading settings :: url = jar:file:/opt/anaconda3/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 79ms :: artifacts dl 2ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-f3c61268-0cef-4c9b-bb9a-0bbdddbd06d5
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/2ms)
25/04/26 23:16:14 WARN NativeCodeLoader: Unable to load native-

/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/scripts/preprocess_with_spark.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().strftime("%H%M%S")


25/04/26 23:16:14 INFO SparkContext: Running Spark version 3.5.5
25/04/26 23:16:14 INFO SparkContext: OS info Mac OS X, 15.3.2, aarch64
25/04/26 23:16:14 INFO SparkContext: Java version 11.0.26
25/04/26 23:16:14 INFO ResourceUtils: ==============================================================
25/04/26 23:16:14 INFO ResourceUtils: No custom resources configured for spark.driver.
25/04/26 23:16:14 INFO ResourceUtils: ==============================================================
25/04/26 23:16:14 INFO SparkContext: Submitted application: BTC Preprocessing with Aggregation
25/04/26 23:16:14 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> name: offHeap, amount: 0, script: , vendor: ), task resources: Map(cpus -> name: cpus, amount: 1.0)
25/04/26 23:16:14 INFO ResourceProfile: Limiting resource is cpu
25/04/26 23:16:14 INFO ResourceProfile

Traceback (most recent call last):
  File "/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/scripts/preprocess_with_spark.py", line 46, in <module>
    main()
  File "/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/scripts/preprocess_with_spark.py", line 20, in main
    df = spark.read.json(S3_INPUT_PATH)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/sql/readwriter.py", line 425, in json
  File "/opt/anaconda3/lib/python3.12/site-packages/pyspark/python/lib/py4j-0.10.9.7-src.zip/py4j/java_gateway.py", line 1322, in __call__
  File "/opt/anaconda3/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/errors/exceptions/captured.py", line 18

25/04/26 23:16:17 INFO SparkContext: Invoking stop() from shutdown hook
25/04/26 23:16:17 INFO SparkContext: SparkContext is stopping with exitCode 0.
25/04/26 23:16:17 INFO SparkUI: Stopped Spark web UI at http://10.0.0.124:4040
25/04/26 23:16:17 INFO MapOutputTrackerMasterEndpoint: MapOutputTrackerMasterEndpoint stopped!
25/04/26 23:16:17 INFO MemoryStore: MemoryStore cleared
25/04/26 23:16:17 INFO BlockManager: BlockManager stopped
25/04/26 23:16:17 INFO BlockManagerMaster: BlockManagerMaster stopped
25/04/26 23:16:17 INFO OutputCommitCoordinator$OutputCommitCoordinatorEndpoint: OutputCommitCoordinator stopped!
25/04/26 23:16:17 INFO SparkContext: Successfully stopped SparkContext
25/04/26 23:16:17 INFO ShutdownHookManager: Shutdown hook called
25/04/26 23:16:17 INFO ShutdownHookManager: Deleting directory /private/var/folders/xb/xbx1vhmd15qdms2yvhk1668m0000gn/T/spark-20e9c1af-9a4f-477c-9554-6891a1846655
25/04/26 23:16:17 INFO ShutdownHookManager: Deleting directory /private/var/fol

In [12]:
# Step 3: Run anomaly explanation script
try:
    subprocess.run(["python", "../scripts/anomaly_explainer.py"], check=True)
    print("Anomaly explanation completed successfully.")
except subprocess.CalledProcessError as e:
    print("Error occurred during anomaly explanation:", e)

Error occurred during anomaly explanation: Command '['python', '../scripts/anomaly_explainer.py']' returned non-zero exit status 1.


/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/notebooks/../scripts/anomaly_explainer.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  datetime.utcnow().strftime("%H%M%S")
Traceback (most recent call last):
  File "/Users/chitturi/Documents/DATA_605/Project/tutorials1/DATA605/Spring2025/projects/TutorTask135_Spring2025_Real_Time_Bitcoin_Transaction_Anomaly_Detection_with_Anthropic_Claude/notebooks/../scripts/anomaly_explainer.py", line 23, in <module>
    files = fs.glob(S3_INPUT_PARQUET.replace("s3://", ""))
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.12/site-packages/fsspec/asyn.py", line 118, in wrapper
    return sync(self.loop, func, *args, **kw